# 可视化：让数据说话
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：13_实用工具 | 源文件：可视化.py | 核心功能：matplotlib、seaborn、图表类型

## 概述

数据可视化帮助理解数据分布、发现模式、展示结果。matplotlib 是基础，seaborn 是高级封装。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd

## 数学原理

### 1. 描述性统计

**均值**：$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$

**中位数**：排序后第 $\lceil n/2 \rceil$ 个值

**方差**：$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$

**分位数**：第 $p$ 分位数 $Q_p$ 满足 $P(X \leq Q_p) = p$

### 2. 相关性矩阵

皮尔逊相关系数矩阵 $R \in \mathbb{R}^{d \times d}$：

$$R_{jk} = \frac{\text{Cov}(x_j, x_k)}{\sigma_j \sigma_k} = \frac{\sum_i(x_{ij}-\bar{x}_j)(x_{ik}-\bar{x}_k)}{\sqrt{\sum_i(x_{ij}-\bar{x}_j)^2 \cdot \sum_i(x_{ik}-\bar{x}_k)^2}}$$

$|R_{jk}| = 1$ 表示完美线性关系，$R_{jk} = 0$ 表示无线性相关。

### 3. 文本直方图

用字符密度近似数据分布：

$$\text{bar}_b = \text{repeat}(\text{"\#"}, \text{round}(n_b / n_{max} \times \text{width}))$$

其中 $n_b$ 是第 $b$ 个桶的样本数。

### 4. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `df.describe()` | 均值、标准差、分位数等 |
| `df.corr()` | 皮尔逊相关系数矩阵 $R$ |
| `np.histogram(data, bins=20)` | 将数据分到 20 个桶，统计 $n_b$ |
| `value_counts(normalize=True)` | 频率分布 $P(x = c)$ |

### 1. 数据表格展示

运行 1. 数据表格展示 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. 数据表格展示")
print("=" * 60)

np.random.seed(42)
n = 200
df = pd.DataFrame({
    '年龄': np.random.randint(18, 65, n),
    '收入': np.random.normal(50000, 15000, n).astype(int),
# --- 数值计算 ---
    '消费': np.random.normal(5000, 2000, n).round(2),
    '满意度': np.random.choice(['低', '中', '高'], n, p=[0.2, 0.5, 0.3]),
    '是否会员': np.random.choice([0, 1], n, p=[0.4, 0.6])
})

print(f"数据集概览:")
print(f"  总行数: {len(df)}")
print(f"  总列数: {len(df.columns)}")
print(f"\n前10行数据:")
print(df.head(10).to_string(index=True))

### 2. 文本直方图

运行 2. 文本直方图 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. 文本直方图")
print("=" * 60)

def text_histogram(data, bins=10, width=40):
    """用文本绘制直方图"""
    counts, edges = np.histogram(data, bins=bins)
    max_count = counts.max()
    labels = []

    for i in range(len(counts)):
        low, high = edges[i], edges[i+1]
        bar_len = int(counts[i] / max_count * width) if max_count > 0 else 0
        bar = '#' * bar_len
        label = f"  [{low:>8.1f}, {high:>8.1f}) |{bar:<{width}} {counts[i]:>4}"
# --- 继续 ---
        labels.append(label)

    return '\n'.join(labels)

print("年龄分布:")
print(text_histogram(df['年龄'], bins=8))
print(f"\n年龄统计:")
print(f"  均值: {df['年龄'].mean():.1f}, 中位数: {df['年龄'].median():.1f}")
print(f"  标准差: {df['年龄'].std():.1f}")
# --- 输出结果 ---
print(f"  范围: [{df['年龄'].min()}, {df['年龄'].max()}]")

print("\n收入分布:")
print(text_histogram(df['收入'], bins=8))

### 3. 分类变量频率表

在分类任务上训练模型并评估性能。

In [ ]:
print("\n" + "=" * 60)
print("3. 分类变量频率表")
print("=" * 60)

def text_bar(value_counts, width=30):
    """用文本绘制水平条形图"""
    max_val = value_counts.max()
    lines = []
    for label, count in value_counts.items():
# --- 条件判断 ---
        bar_len = int(count / max_val * width) if max_val > 0 else 0
        bar = '#' * bar_len
        pct = count / value_counts.sum() * 100
        lines.append(f"  {str(label):<8} |{bar:<{width}} {count:>4} ({pct:>5.1f}%)")
    return '\n'.join(lines)

print("满意度分布:")
print(text_bar(df['满意度'].value_counts().sort_index()))

print("\n是否会员分布:")
print(text_bar(df['是否会员'].value_counts().sort_index()))

### 4. 相关性矩阵 (文本展示)

运行 4. 相关性矩阵 (文本展示) 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("4. 相关性矩阵")
print("=" * 60)

numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

# 文本形式展示相关性
print(f"{'':>10}", end="")
for col in numeric_cols:
    print(f"{col:>10}", end="")
print()

for i, row_name in enumerate(numeric_cols):
    print(f"{row_name:>10}", end="")
    for j, col_name in enumerate(numeric_cols):
        val = corr.iloc[i, j]
        # 用符号表示相关性强度
        if abs(val) > 0.7:
            marker = "*"
        elif abs(val) > 0.3:
            marker = "+"
        else:
# --- 赋值/配置 ---
            marker = " "
        print(f"{val:>9.3f}{marker}", end="")
    print()

print("\n图例: *=强相关(>0.7), +=中等相关(>0.3)")

### 5. 箱线图文本表示

运行 5. 箱线图文本表示 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. 箱线图文本表示")
print("=" * 60)

def text_boxplot(data, name=""):
    """文本形式展示箱线图信息"""
    q1 = np.percentile(data, 25)
    q2 = np.percentile(data, 50)
    q3 = np.percentile(data, 75)
# --- 赋值/配置 ---
    iqr = q3 - q1
    lower_fence = max(data.min(), q1 - 1.5 * iqr)
    upper_fence = min(data.max(), q3 + 1.5 * iqr)
    outliers = np.sum((data < lower_fence) | (data > upper_fence))

    # 简单的文本箱线图
    min_val, max_val = data.min(), data.max()
    scale = 50  # 字符宽度
    rng = max_val - min_val if max_val > min_val else 1

    def pos(val):
        return int((val - min_val) / rng * scale)

    line = [' '] * (scale + 1)
    # 画箱体
    for i in range(pos(q1), pos(q3) + 1):
        line[i] = '-'
    line[pos(q2)] = '|'

    print(f"\n  {name}:")
    print(f"  Min={min_val:.2f}  Q1={q1:.2f}  Median={q2:.2f}  Q3={q3:.2f}  Max={max_val:.2f}")
    print(f"  IQR={iqr:.2f}, 异常值数={outliers}")
    print(f"  [{''.join(line)}]")

for col in ['年龄', '收入', '消费']:
    text_boxplot(df[col].values, col)

### 6. 交叉频率表

运行 6. 交叉频率表 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("6. 交叉频率表 (分类变量)")
print("=" * 60)

# 满意度 vs 是否会员
cross = pd.crosstab(df['满意度'], df['是否会员'], margins=True)
cross.columns = ['非会员', '会员', '合计']
cross.index.name = '满意度'

print(cross.to_string())

print("\n会员比例 (按满意度):")
for satisfaction in ['低', '中', '高']:
    subset = df[df['满意度'] == satisfaction]
    member_rate = subset['是否会员'].mean() * 100
    bar_len = int(member_rate / 100 * 20)
# --- 赋值/配置 ---
    bar = '#' * bar_len
    print(f"  {satisfaction}: {bar} {member_rate:.1f}%")

### 7. 数据摘要输出

运行 7. 数据摘要输出 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("7. 综合数据摘要")
print("=" * 60)

print(f"数据集: {df.shape[0]}行 x {df.shape[1]}列")
print(f"\n数值特征摘要:")
for col in df.select_dtypes(include=[np.number]).columns:
    series = df[col]
    print(f"  {col}:")
# --- 输出结果 ---
    print(f"    均值={series.mean():.2f}, 中位数={series.median():.2f}, "
          f"std={series.std():.2f}, 范围=[{series.min()}, {series.max()}]")

print(f"\n分类特征摘要:")
for col in df.select_dtypes(include='object').columns:
    vc = df[col].value_counts()
    top = vc.index[0]
    print(f"  {col}: {len(vc)}个类别, 最频繁=\"{top}\"({vc.iloc[0]}次, {vc.iloc[0]/len(df)*100:.1f}%)")

## 关键代码解释

```python
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(confusion_matrix, annot=True)
sns.pairplot(df, hue="target")
```

## 注意事项

1. 图表要有标题、坐标轴标签、图例
2. 颜色选择要考虑色盲友好
3. 保存图片用 `savefig(dpi=300)`

## 总结与延伸

以上代码展示了 **可视化** 的完整流程。

**解读要点**：
- 关注工具的 **输入输出格式** 是否正确
- 观察数据加载和预处理的效率
- 检查模型保存/加载后的一致性

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Plotly**：交互式可视化
- **TensorBoard**：训练过程可视化
- **SHAP 图**：模型可解释性可视化
